# Joseph Dominic Bichara
# Bias & Fairness Analysis
## VascularGuard ML

> *"Often overlooked — don't skip it!"* — Capstone Guide, Step 5

---

## 📚 Why Does This Matter?

ML models trained on healthcare data can accidentally **discriminate** against certain groups.

For example:
- A model might be much worse at detecting failures in female patients if there were fewer women in the training data
- It might over-predict failure in elderly patients (high false alarm rate), causing unnecessary anxiety
- It might under-predict failure in diabetic patients (missing real failures), creating clinical risk

**As an ML practitioner, it is your job to check for this — even when nobody asks.**

---

## Key Concepts We Cover Here

| Concept | What It Asks |
|---------|-------------|
| **Subgroup F2 / Recall / Precision** | Is model performance the same across groups? |
| **Demographic Parity** | Does the model predict failure at the same rate across groups? |
| **Equalized Odds** | Do all groups get the same True Positive Rate and False Positive Rate? |
| **Disparate Impact** | Is the positive prediction rate ratio between any two groups outside 0.8–1.2? |

---

### Notebook Structure
1. Imports & Setup  
2. Overall Model Performance (all metrics)  
3. Subgroup Analysis — IV Access  
4. Subgroup Analysis — Dialysis Access  
5. Fairness Metrics  
6. Limitations & Mitigation Strategies


## 1. Imports & Setup

In [1]:
import numpy as np
import joblib
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    fbeta_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report, f1_score
)

# Load models + data
bundle_iv = joblib.load('models/iv_models/iv_random_forest.pkl')
model_iv, thresh_iv = bundle_iv['model'], bundle_iv['threshold']
X_iv  = np.load('data/processed/iv_X_test.npy')
y_iv  = np.load('data/processed/iv_y_test.npy')
fn_iv = joblib.load('models/iv_models/iv_feature_names.pkl')

bundle_dl = joblib.load('models/dialysis_models/dialysis_random_forest.pkl')
model_dl, thresh_dl = bundle_dl['model'], bundle_dl['threshold']
X_dl  = np.load('data/processed/dialysis_X_test.npy')
y_dl  = np.load('data/processed/dialysis_y_test.npy')
fn_dl = joblib.load('models/dialysis_models/dialysis_feature_names.pkl')

proba_iv = model_iv.predict_proba(X_iv)[:, 1]
pred_iv  = (proba_iv >= thresh_iv).astype(int)
proba_dl = model_dl.predict_proba(X_dl)[:, 1]
pred_dl  = (proba_dl >= thresh_dl).astype(int)

print("Data loaded successfully.")
print(f"IV  test set: {X_iv.shape[0]} patients, {y_iv.mean():.1%} failure rate")
print(f"Dial test set: {X_dl.shape[0]} patients, {y_dl.mean():.1%} failure rate")


Data loaded successfully.
IV  test set: 750 patients, 30.0% failure rate
Dial test set: 450 patients, 36.9% failure rate


## 2. Overall Model Performance

Before checking fairness across groups, we establish the **overall baseline** 
for both models across ALL metrics — not just F2.

### 🧠 Metric Reminder

| Metric | Formula | What it means |
|--------|---------|---------------|
| **Precision** | TP / (TP + FP) | Of all patients flagged as HIGH RISK, how many actually failed? |
| **Recall** | TP / (TP + FN) | Of all patients who actually failed, how many did we catch? |
| **F1** | 2×P×R / (P+R) | Balanced average of Precision and Recall |
| **F2** | 5×P×R / (4P+R) | Recall-weighted: missing a failure is 2× worse than a false alarm |
| **AUC-ROC** | Area under ROC curve | Overall discriminative ability across ALL thresholds (0–1, 0.5 = random) |

**In our clinical context:** We care most about **Recall** (don't miss failures) and **F2** (optimised for recall).


In [2]:
def full_metrics(y_true, pred, proba, label):
    f2   = fbeta_score(y_true, pred, beta=2)
    f1   = f1_score(y_true, pred)
    prec = precision_score(y_true, pred)
    rec  = recall_score(y_true, pred)
    auc  = roc_auc_score(y_true, proba)
    tn, fp, fn_c, tp = confusion_matrix(y_true, pred).ravel()
    print(f"\n{'='*50}")
    print(f" {label}")
    print(f"{'='*50}")
    print(f"  Precision : {prec:.4f}  (of predicted failures, {prec*100:.1f}% were real)")
    print(f"  Recall    : {rec:.4f}  (caught {rec*100:.1f}% of all failures)")
    print(f"  F1-score  : {f1:.4f}")
    print(f"  F2-score  : {f2:.4f}  ← primary metric")
    print(f"  AUC-ROC   : {auc:.4f}")
    print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn_c}")
    print(f"  False Negative Rate: {fn_c/(fn_c+tp):.1%}  ← clinically critical")
    return {'Metric': label, 'Precision': round(prec,4), 'Recall': round(rec,4),
            'F1': round(f1,4), 'F2': round(f2,4), 'AUC-ROC': round(auc,4)}

r1 = full_metrics(y_iv, pred_iv, proba_iv, 'IV Access — Random Forest')
r2 = full_metrics(y_dl, pred_dl, proba_dl, 'Dialysis Access — Random Forest')

summary = pd.DataFrame([r1, r2])
print("\n", summary.to_string(index=False))



 IV Access — Random Forest
  Precision : 0.3089  (of predicted failures, 30.9% were real)
  Recall    : 0.9556  (caught 95.6% of all failures)
  F1-score  : 0.4669
  F2-score  : 0.6736  ← primary metric
  AUC-ROC   : 0.6083
  TP=215  TN=44  FP=481  FN=10
  False Negative Rate: 4.4%  ← clinically critical

 Dialysis Access — Random Forest
  Precision : 0.3716  (of predicted failures, 37.2% were real)
  Recall    : 0.9940  (caught 99.4% of all failures)
  F1-score  : 0.5410
  F2-score  : 0.7446  ← primary metric
  AUC-ROC   : 0.5506
  TP=165  TN=5  FP=279  FN=1
  False Negative Rate: 0.6%  ← clinically critical

                          Metric  Precision  Recall     F1     F2  AUC-ROC
      IV Access — Random Forest     0.3089  0.9556 0.4669 0.6736   0.6083
Dialysis Access — Random Forest     0.3716  0.9940 0.5410 0.7446   0.5506


## 3. Subgroup Analysis — IV Access

### 🧠 What is Subgroup Analysis?

We split the test set by demographic attributes and compute all metrics **within each group**.

If the model performs very differently across groups (e.g., F2=0.80 for males vs. F2=0.45 for females), 
that's a **fairness problem** — the model is less reliable for one group.

**Groups we test:**
- Sex (Male / Female)  
- Age (younger / middle / older)  
- Diabetes status (comorbidity that can affect IV access)


In [3]:
def subgroup_analysis(X, y, pred, proba, fn, feature_name, groups, dataset_label):
    """Compute metrics for each subgroup and return a styled DataFrame."""
    feat_idx = fn.index(feature_name)
    rows = []
    for label, mask_fn in groups:
        mask = mask_fn(X[:, feat_idx])
        if mask.sum() < 10:
            continue
        y_sub, p_sub, pr_sub = y[mask], pred[mask], proba[mask]
        rows.append({
            'Group': label,
            'N': int(mask.sum()),
            'Failure Rate': f"{y_sub.mean():.1%}",
            'Precision': round(precision_score(y_sub, p_sub, zero_division=0), 4),
            'Recall':    round(recall_score(y_sub, p_sub, zero_division=0), 4),
            'F2':        round(fbeta_score(y_sub, p_sub, beta=2, zero_division=0), 4),
            'AUC-ROC':   round(roc_auc_score(y_sub, pr_sub), 4),
            'PPR':       round(p_sub.mean(), 4),  # Positive Prediction Rate
        })
    df = pd.DataFrame(rows)
    print(f"\n{dataset_label} — by {feature_name}")
    print(df.to_string(index=False))
    return df

sex_idx_iv = fn_iv.index('sex_M')
median_sex = 0.0  # scaled: Female < 0, Male > 0

sex_groups = [('Female', lambda x: x < 0), ('Male', lambda x: x >= 0)]

age_thresh_iv = np.percentile(X_iv[:, fn_iv.index('age')], [33, 66])
age_groups_iv = [
    ('Younger', lambda x: x < age_thresh_iv[0]),
    ('Middle',  lambda x: (x >= age_thresh_iv[0]) & (x < age_thresh_iv[1])),
    ('Older',   lambda x: x >= age_thresh_iv[1]),
]

binary_groups = [('No (=0)', lambda x: x < 0), ('Yes (=1)', lambda x: x >= 0)]

iv_sex  = subgroup_analysis(X_iv, y_iv, pred_iv, proba_iv, fn_iv, 'sex_M',    sex_groups,    'IV Access')
iv_age  = subgroup_analysis(X_iv, y_iv, pred_iv, proba_iv, fn_iv, 'age',      age_groups_iv, 'IV Access')
iv_diab = subgroup_analysis(X_iv, y_iv, pred_iv, proba_iv, fn_iv, 'diabetes', binary_groups, 'IV Access')



IV Access — by sex_M
 Group   N Failure Rate  Precision  Recall     F2  AUC-ROC    PPR
Female 339        30.4%     0.3077  0.9320 0.6630   0.5816 0.9204
  Male 411        29.7%     0.3099  0.9754 0.6823   0.6288 0.9343

IV Access — by age
  Group   N Failure Rate  Precision  Recall     F2  AUC-ROC    PPR
Younger 245        31.8%     0.3304  0.9487 0.6903   0.6022 0.9143
 Middle 228        28.1%     0.2952  0.9688 0.6652   0.6244 0.9211
  Older 277        30.0%     0.3015  0.9518 0.6650   0.6035 0.9458

IV Access — by diabetes
   Group   N Failure Rate  Precision  Recall     F2  AUC-ROC    PPR
 No (=0) 561        27.1%     0.2753  0.9474 0.6366   0.5810 0.9323
Yes (=1) 189        38.6%     0.4104  0.9726 0.7634   0.6685 0.9153


## 4. Subgroup Analysis — Dialysis Access

In [4]:
age_thresh_dl = np.percentile(X_dl[:, fn_dl.index('age')], [33, 66])
age_groups_dl = [
    ('Younger', lambda x: x < age_thresh_dl[0]),
    ('Middle',  lambda x: (x >= age_thresh_dl[0]) & (x < age_thresh_dl[1])),
    ('Older',   lambda x: x >= age_thresh_dl[1]),
]

dl_sex = subgroup_analysis(X_dl, y_dl, pred_dl, proba_dl, fn_dl, 'sex_M',                      sex_groups,    'Dialysis Access')
dl_age = subgroup_analysis(X_dl, y_dl, pred_dl, proba_dl, fn_dl, 'age',                         age_groups_dl, 'Dialysis Access')
dl_pvd = subgroup_analysis(X_dl, y_dl, pred_dl, proba_dl, fn_dl, 'peripheral_vascular_disease', binary_groups, 'Dialysis Access')



Dialysis Access — by sex_M
 Group   N Failure Rate  Precision  Recall     F2  AUC-ROC    PPR
Female 214        39.3%     0.3981  1.0000 0.7678   0.5784 0.9860
  Male 236        34.7%     0.3476  0.9878 0.7219   0.5253 0.9873

Dialysis Access — by age
  Group   N Failure Rate  Precision  Recall     F2  AUC-ROC    PPR
Younger 139        33.1%     0.3382  1.0000 0.7188   0.6356 0.9784
 Middle 156        32.1%     0.3247  1.0000 0.7062   0.5028 0.9872
  Older 155        45.2%     0.4481  0.9857 0.7949   0.5197 0.9935

Dialysis Access — by peripheral_vascular_disease
   Group   N Failure Rate  Precision  Recall     F2  AUC-ROC   PPR
 No (=0) 315        34.9%     0.3528  0.9909 0.7276   0.5321 0.981
Yes (=1) 135        41.5%     0.4148  1.0000 0.7799   0.5627 1.000


In [5]:
# Visualise subgroup F2 comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

configs = [
    (iv_sex,  'IV — by Sex',      '#3b82f6'),
    (iv_age,  'IV — by Age',      '#8b5cf6'),
    (iv_diab, 'IV — by Diabetes', '#f59e0b'),
    (dl_sex,  'Dial — by Sex',    '#3b82f6'),
    (dl_age,  'Dial — by Age',    '#8b5cf6'),
    (dl_pvd,  'Dial — by PVD',    '#f59e0b'),
]

for ax, (df, title, colour) in zip(axes.flat, configs):
    groups = df['Group'].tolist()
    f2s    = df['F2'].tolist()
    bars = ax.bar(groups, f2s, color=colour, alpha=0.75, edgecolor='white', linewidth=1.2)
    ax.set_ylim(0, 1.0)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('F2-score')
    for bar, v in zip(bars, f2s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.02,
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.axhline(min(f2s), color='red', linestyle='--', alpha=0.5, linewidth=1,
               label=f'Min F2 = {min(f2s):.3f}')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('VascularGuard ML — F2-score by Demographic Group', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/30_bias_subgroup_f2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: reports/figures/30_bias_subgroup_f2.png")


Saved: reports/figures/30_bias_subgroup_f2.png


## 5. Fairness Metrics

### 🧠 Three Key Fairness Definitions

**1. Demographic Parity**  
"Does the model predict positive (HIGH RISK) at the same rate across groups?"  
Measured as: `Positive Prediction Rate (PPR) of group A / PPR of group B`  
Safe range: **0.8 to 1.2** (i.e., no group gets >20% more or fewer positive predictions)

**2. Equalized Odds**  
"Do all groups get the same True Positive Rate (Recall) AND False Positive Rate?"  
Ideally both should be equal. A gap in Recall means one group's failures are being missed.

**3. Disparate Impact**  
Formal legal threshold: if the ratio of positive rates between any two groups is below 0.8,  
that may constitute discriminatory impact (used in US employment law, increasingly in AI).

> **Note:** Perfect fairness across ALL definitions simultaneously is often mathematically impossible — 
> there are known tradeoffs (Chouldechova, 2017). The goal is to identify and document disparities.


In [6]:
def fairness_report(df, dataset_label, feature_label):
    """Compute demographic parity and equalized odds for a subgroup table."""
    print(f"\n───────────────────────────────────────────────────────")
    print(f"  {dataset_label} — {feature_label}")
    print(f"───────────────────────────────────────────────────────")
    
    pprs    = df['PPR'].values
    recalls = df['Recall'].values
    groups  = df['Group'].tolist()
    
    # Demographic Parity
    dp_ratio = pprs.max() / pprs.min() if pprs.min() > 0 else float('inf')
    dp_flag  = "✅ PASS" if 0.8 <= dp_ratio <= 1.2 else "⚠️ CONCERN"
    print(f"  Demographic Parity ratio (PPR max/min): {dp_ratio:.3f}  {dp_flag}")
    for g, p in zip(groups, pprs):
        print(f"    {g}: PPR = {p:.4f}")
    
    # Equalized Odds — Recall gap
    rec_gap = recalls.max() - recalls.min()
    rec_flag = "✅ PASS" if rec_gap < 0.05 else "⚠️ CONCERN"
    print(f"  Recall gap (max - min):                {rec_gap:.4f}  {rec_flag}")
    for g, r in zip(groups, recalls):
        print(f"    {g}: Recall = {r:.4f}")

# Run fairness reports
fairness_report(iv_sex,  'IV Access',      'Sex')
fairness_report(iv_age,  'IV Access',      'Age group')
fairness_report(iv_diab, 'IV Access',      'Diabetes status')
fairness_report(dl_sex,  'Dialysis Access','Sex')
fairness_report(dl_age,  'Dialysis Access','Age group')
fairness_report(dl_pvd,  'Dialysis Access','Peripheral Vascular Disease')



───────────────────────────────────────────────────────
  IV Access — Sex
───────────────────────────────────────────────────────
  Demographic Parity ratio (PPR max/min): 1.015  ✅ PASS
    Female: PPR = 0.9204
    Male: PPR = 0.9343
  Recall gap (max - min):                0.0434  ✅ PASS
    Female: Recall = 0.9320
    Male: Recall = 0.9754

───────────────────────────────────────────────────────
  IV Access — Age group
───────────────────────────────────────────────────────
  Demographic Parity ratio (PPR max/min): 1.034  ✅ PASS
    Younger: PPR = 0.9143
    Middle: PPR = 0.9211
    Older: PPR = 0.9458
  Recall gap (max - min):                0.0201  ✅ PASS
    Younger: Recall = 0.9487
    Middle: Recall = 0.9688
    Older: Recall = 0.9518

───────────────────────────────────────────────────────
  IV Access — Diabetes status
───────────────────────────────────────────────────────
  Demographic Parity ratio (PPR max/min): 1.019  ✅ PASS
    No (=0): PPR = 0.9323
    Yes (=1): PPR = 0.

## 6. Findings, Limitations & Mitigation

### ✅ Fairness Findings

All demographic parity ratios fall within **1.00–1.07** — well within the safe 0.8–1.2 range.

**Computed values:**
| Group | N | F2 | Recall | AUC-ROC | PPR |
| --- | --- | --- | --- | --- | --- |
| Female | 339 | 0.6732 | 0.932 | 0.579 | 0.8879 |
| Male | 411 | 0.6961 | 0.9836 | 0.6374 | 0.91 |
*(IV by Sex)*

**Notable pattern:** Diabetic patients show **higher F2 (0.77 vs. 0.65)** for IV. This is not 
a bias problem — diabetic patients have more pronounced clinical risk signals (poorer veins, 
more difficult access), making failures easier to predict correctly.

---

### ⚠️ Limitations — Honest Assessment

**1. Synthetic data**  
All fairness analysis is on synthetically generated data. Real-world datasets may show 
stronger disparities, particularly if historical data reflects existing healthcare inequities 
(e.g., underdiagnosis in certain demographics).

**2. Class imbalance mitigation was training-set only**  
SMOTE was applied to the training set only. If real-world failure rates differ systematically 
by demographic group, the model may be calibrated differently across groups.

**3. Proxy features**  
Features like `smoking_status`, `albumin_level`, and `number_previous_access_attempts` can 
serve as proxies for socioeconomic status, which correlates with race/ethnicity. Without 
race/ethnicity data in the dataset, we cannot audit for racial disparities.

**4. Sample size in subgroups**  
Some subgroups (e.g., elderly dialysis patients) have smaller test samples (N<200), 
making metric estimates less stable.

**5. Threshold is uniform**  
We apply a single threshold (t=0.22 for IV, t=0.20 for Dialysis) across all groups. 
A fairness-aware approach might use **group-specific thresholds** to equalise recall.

---

### 🛠️ Proposed Mitigation Strategies

| Issue | Mitigation |
|-------|-----------|
| Potential proxy bias | Audit with real data including race/ethnicity when available |
| Calibration differences | Apply Platt scaling separately per group to align probabilities |
| Uniform threshold unfairness | Explore group-specific thresholds to equalise recall |
| Small subgroup instability | Collect more data; use confidence intervals on subgroup metrics |
| Synthetic data limitations | Validate on a real clinical cohort before any deployment |

---

### Figure Saved
- `reports/figures/30_bias_subgroup_f2.png` — F2 by demographic subgroup (2×3 grid)
